# Grant App Permissions

Grants required permissions on the Databricks App after deploy:
1. **M2M SPN → CAN_USE** — required so the MCP HTTP connection can authenticate through the app's auth sidecar

App-level permissions use the workspace Permissions API (not SQL DDL), so this is a Python notebook.

Run by `deploy.sh` Phase 2 via `databricks bundle run app_post_deploy_setup`.
Also safe to run manually — `update_permissions` is idempotent (PATCH).

In [0]:
%pip install --upgrade databricks-sdk
dbutils.library.restartPython()

In [0]:
dbutils.widgets.text("app_name", "", "App name")
dbutils.widgets.text("scope_name", "", "Secret scope")
dbutils.widgets.text("client_id_key", "", "M2M client ID key in scope")

app_name = dbutils.widgets.get("app_name")
scope_name = dbutils.widgets.get("scope_name")
client_id_key = dbutils.widgets.get("client_id_key")

assert app_name, "app_name is required"
assert scope_name, "scope_name is required"
assert client_id_key, "client_id_key is required"

print(f"App:            {app_name}")
print(f"Scope:          {scope_name}")
print(f"Client ID key:  {client_id_key}")

In [0]:
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()

# Read M2M SPN client_id from secret scope
m2m_client_id = dbutils.secrets.get(scope=scope_name, key=client_id_key)
print(f"M2M SPN client_id: {m2m_client_id}")

# Look up SPN by applicationId
spn = None
for sp in w.service_principals.list(filter=f'applicationId eq "{m2m_client_id}"'):
    spn = sp
    break

if not spn:
    raise ValueError(f"Service principal not found for client_id={m2m_client_id}")

print(f"Found SPN: {spn.display_name} (id={spn.id}, appId={spn.application_id})")

In [0]:
from databricks.sdk.service.iam import AccessControlRequest, PermissionLevel

result = w.apps.update_permissions(
    app_name=app_name,
    access_control_list=[
        AccessControlRequest(
            service_principal_name=spn.display_name,
            permission_level=PermissionLevel.CAN_USE,
        )
    ],
)
print(f"\u2713 Granted CAN_USE to {spn.display_name} on {app_name}")

In [0]:
import json

perms = w.apps.get_permissions(app_name)
grants = []
for acl in perms.access_control_list or []:
    principal = acl.user_name or acl.group_name or acl.service_principal_name or "unknown"
    levels = [p.permission_level.value for p in (acl.all_permissions or [])]
    grants.append({"principal": principal, "permissions": levels})
    print(f"  {principal}: {levels}")

dbutils.notebook.exit(json.dumps({"app_name": app_name, "grants": grants}))